In [38]:
import pandas as pd

df=pd.read_csv("/content/sample_data/100_Unique_QA_Dataset.csv")
df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


In [39]:
# Tokenization

def tokenize(text):
  text=text.lower()
  text=text.replace("?","")
  text=text.replace("`","")
  return text.split()

In [40]:
# vocabulary

vocab={"<UNK":0}


In [41]:
# convert word into numerical indices

def build_vocab(row):
  tokenized_question=tokenize(row["question"])
  tokenzied_answer=tokenize(row["answer"])

  merged_tokens=tokenized_question + tokenzied_answer

  for token in merged_tokens:
    if token not in vocab:
      vocab[token]=len(vocab)



In [42]:
df.apply(build_vocab,axis=1)

,0
0,None
1,None
2,None
3,None
4,None
...,...
85,None
86,None
87,None
88,None


In [43]:
len(vocab)

326

In [44]:
def text_to_indices(text,vocab):
  indexed_text=[]

  for token in tokenize(text):
    if token in vocab:
      indexed_text.append(vocab[token])
    else:
      indexed_text.append(vocab["<UNK>"])

  return indexed_text

In [45]:
import torch
from torch.utils.data import Dataset, DataLoader

class QADataset(Dataset):
    def __init__(self, df, vocab):
        self.df = df
        self.vocab = vocab

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        numerical_question = text_to_indices(
            self.df.iloc[index]["question"],
            self.vocab
        )

        numerical_answer = text_to_indices(
            self.df.iloc[index]["answer"],
            self.vocab
        )

        return (
            torch.tensor(numerical_question),
            torch.tensor(numerical_answer[0])
        )

In [46]:
from torch.utils.data import Dataset, DataLoader

dataset = QADataset(df, vocab)

dataloader = DataLoader(
    dataset,
    batch_size=1,
    shuffle=True
)

In [47]:
import torch.nn as nn

class SimpleRNN(nn.Module):

  def __init__(self,vocab_size):
    super().__init__()
    self.embedding=nn.Embedding(vocab_size,embedding_dim=50)
    self.rnn=nn.RNN(50,64)
    nn.Linear(64,vocab_size)
    self.fc=nn.Linear(64,vocab_size)

  def forward(self,question):
    embedded_question=self.embedding(question)
    hidden,final=self.rnn(embedded_question)
    output=self.fc(final)
    return output


In [49]:
class SimpleRNN(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, 50)
        self.rnn = nn.RNN(50, 64, batch_first=True)
        self.fc = nn.Linear(64, vocab_size)

    def forward(self, question):
        embedded_question = self.embedding(question)

        hidden, final = self.rnn(embedded_question)

        output = self.fc(final.squeeze(0))

        return output

In [50]:
learning_rate = 0.001
epochs = 20

model = SimpleRNN(len(vocab))

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=learning_rate
)

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for question, answer in dataloader:

        optimizer.zero_grad()

        # Forward pass
        output = model(question)

        # Calculate loss
        loss = criterion(output, answer)

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(dataloader)

    print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}")

Epoch [1/20], Loss: 5.8490
Epoch [2/20], Loss: 5.0803
Epoch [3/20], Loss: 4.2073
Epoch [4/20], Loss: 3.5402
Epoch [5/20], Loss: 2.9666
Epoch [6/20], Loss: 2.4346
Epoch [7/20], Loss: 1.9453
Epoch [8/20], Loss: 1.5164
Epoch [9/20], Loss: 1.1652
Epoch [10/20], Loss: 0.8933
Epoch [11/20], Loss: 0.6970
Epoch [12/20], Loss: 0.5445
Epoch [13/20], Loss: 0.4310
Epoch [14/20], Loss: 0.3553
Epoch [15/20], Loss: 0.2907
Epoch [16/20], Loss: 0.2418
Epoch [17/20], Loss: 0.2044
Epoch [18/20], Loss: 0.1739
Epoch [19/20], Loss: 0.1489
Epoch [20/20], Loss: 0.1294


In [57]:
def predict(model, question, threshold=0.5):
  numerical_question=text_to_indices(question,vocab)

  # convert it into tensor
  question_tensor=torch.tensor(numerical_question).unsqueeze(0)

  # send to model
  output=model(question_tensor)

  # convert logits to probs
  probs=torch.nn.functional.softmax(output,dim=1)

  # find index of max prob
  value,index=torch.max(probs,dim=1)


  if value<threshold:
    print("I dont know")

  list(vocab.keys())[index]


In [58]:
predict(model,"what is capital  of france")

In [59]:
list(vocab.keys())[7]

'paris'